# SpaceX Falcon 9 Exploratory Data Analysis with SQL
This notebook executes all 10 standard SQL analysis queries on the historical SpaceX launch table (`SPACEXTBL`) using SQLite.

In [1]:
import sqlite3
import pandas as pd

# Load CSV and normalize column names for SQL compatibility
df = pd.read_csv('../data/Spacex.csv')
df.columns = [c.strip().replace(' ', '_').replace('(', '').replace(')', '').replace('__', '_') for c in df.columns]

conn = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', conn, if_exists='replace', index=False)
print('Loaded SPACEXTBL into SQLite in-memory database with', len(df), 'rows.')

Loaded SPACEXTBL into SQLite in-memory database with 101 rows.


## Query 1: Distinct Launch Sites
Names of unique launch sites in the SpaceX dataset.

In [2]:
q1 = pd.read_sql_query('SELECT DISTINCT Launch_Site FROM SPACEXTBL', conn)
q1

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


## Query 2: Launch Sites Beginning with 'CCA'
First 5 launch records where launch site begins with 'CCA'.

In [3]:
q2 = pd.read_sql_query("SELECT Launch_Site FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5", conn)
q2

,Launch_Site
0,CCAFS LC-40
1,CCAFS LC-40
2,CCAFS LC-40
3,CCAFS LC-40
4,CCAFS LC-40


## Query 3: Total Payload Carried by NASA (CRS)
Total payload mass delivered by boosters launched for customer NASA (CRS).

In [4]:
q3 = pd.read_sql_query("SELECT SUM(PAYLOAD_MASS_KG_) AS Total_Payload_Mass FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'", conn)
q3

,Total_Payload_Mass
0,45596


## Query 4: Average Payload Carried by Booster Version F9 v1.1
Average payload mass carried by F9 v1.1 booster versions.

In [5]:
q4 = pd.read_sql_query("SELECT AVG(PAYLOAD_MASS_KG_) AS Avg_Payload_Mass FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'", conn)
q4

,Avg_Payload_Mass
0,2534.666667


## Query 5: Date of First Successful Ground Landing
Earliest successful landing outcome on ground pad.

In [6]:
q5 = pd.read_sql_query("SELECT MIN(Date) AS First_Successful_Ground_Landing FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'", conn)
q5

,First_Successful_Ground_Landing
0,2015-12-22


## Query 6: Boosters with Drone Ship Success & 4000-6000 kg Payload
Booster versions that achieved successful drone ship landing with payload between 4000 and 6000 kg.

In [7]:
q6 = pd.read_sql_query("SELECT Booster_Version, PAYLOAD_MASS_KG_ FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS_KG_ BETWEEN 4000 AND 6000", conn)
q6

,Booster_Version,PAYLOAD_MASS_KG_
0,F9 FT B1022,4696
1,F9 FT B1026,4600
2,F9 FT B1021.2,5300
3,F9 FT B1031.2,5200


## Query 7: Total Successful and Failed Mission Outcomes
Frequency count of successful versus failed mission outcomes.

In [8]:
q7 = pd.read_sql_query('SELECT Mission_Outcome, COUNT(*) AS Count FROM SPACEXTBL GROUP BY Mission_Outcome', conn)
q7

,Mission_Outcome,Count
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


## Query 8: Booster Versions with Maximum Payload Mass
Boosters that carried the maximum payload mass recorded.

In [9]:
q8 = pd.read_sql_query('SELECT Booster_Version, PAYLOAD_MASS_KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS_KG_ = (SELECT MAX(PAYLOAD_MASS_KG_) FROM SPACEXTBL)', conn)
q8

,Booster_Version,PAYLOAD_MASS_KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


## Query 9: Drone Ship Landing Failures in 2015
Records of failed drone ship landings during calendar year 2015.

In [10]:
q9 = pd.read_sql_query("SELECT SUBSTR(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND SUBSTR(Date, 1, 4) = '2015'", conn)
q9

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Query 10: Landing Outcome Rankings (2010-06-04 to 2017-03-20)
Ranking of landing outcomes between specified dates in descending order of frequency.

In [11]:
q10 = pd.read_sql_query("SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count DESC", conn)
q10

,Landing_Outcome,Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
